In [1]:
import numpy as np

In [2]:
## data preparation
def prepare_data_splits(X, y, train_prop = 0.98, dev_prop = 0.01):
    m = X.shape[0]
    shuffled_indices = np.random.permutation(m)
    X_shuffled = X[shuffled_indices]
    y_shuffled = y[shuffled_indices]

    train_end = int(m * train_prop)
    dev_end = train_end + int(m * dev_prop)

    X_train, y_train = X_shuffled[:train_end], y_shuffled[:train_end]
    X_dev, y_dev = X_shuffled[train_end:dev_end], y_shuffled[train_end:dev_end]
    X_test, y_test =  X_shuffled[dev_end:], y_shuffled[dev_end:]
    return X_train, y_train, X_dev, y_dev, X_test, y_test


In [3]:
##normalization features

def normalization_feature(X_train, X_dev, X_test):
    mu = np.mean(X_train, axis = 0, keepdims = True)
    sigma = np.std(X_train, axis = 0, keepdims = True)

    X_train_norm = (X_train - mu) / sigma
    X_dev_norm  = (X_dev - mu) / sigma
    X_test_norm = (X_test - mu) / sigma
    
    return X_train_norm, X_dev_norm, X_test_norm



Input-> Hidden1 (ReLu) -> Hidden2 (ReLu) -> OutPut(Softmax/ Sigmoid)

In [4]:
## initialize parameters naive 

def initialize_parameter_naive(layer_dims):
    parameters = {}
    L = len(layer_dims)

    for l in range(1, L):
        parameters['W' + str(l)] = np.random.randn(layer_dims[l], layer_dims[l-1]) * 0.01
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
    
    return parameters



In [5]:
## forward propagation
def relu(Z): return np.maximum(0, Z)
def softmax(Z):
    exp_Z = np.exp(Z -np.max(Z, axis = 0, keepdims = True))
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

def forward_propagation(X_batch, parameters):
    caches = {}
    A = X_batch.T
    caches['A0'] = A
    L = len(parameters) // 2

    for l in range(1, L):
        Z = np.dot(parameters['W' + str(l)], A)  + parameters['b' + str(l)]
        A = relu(Z)

        caches['Z' + str(l)] = Z
        caches['A' + str(l)] = A

    ZL = np.dot(parameters['W' + str(L)], A) + parameters['b' + str(L)]
    AL = softmax(ZL)
    caches['ZL' + str(L)] = ZL
    caches['AL' + str(L)] = AL

    return AL, caches

In [6]:
# compute cost

def compute_cost(AL, Y_batch):
    m = Y_batch.shape[0]
    Y = Y_batch.T

    # cost = -(1.0 / m) * np.sum(Y * np.log(AL + 1e-8))
    cost = - (1.0 / m) * np.sum(Y * np.log(AL + 1e-8))
    return np.squeeze(cost)
    

In [7]:
## Backward Propagation

def relu_backward(dA, Z):
    dZ = np.array(dA, copy=True)
    dZ[Z<=0] = 0
    return dZ

def backward_propagation(caches, parameters, Y_batch):
    grads = {}
    m = Y_batch.shape[0]
    Y = Y_batch.T
    L = len(parameters) // 2

    dZL = caches['AL' + str(L)] - Y
    grads['dW' + str(L)] = (1.0 / m) * np.dot(dZL, caches['A' + str(L-1)].T)
    grads['db' + str(L)] = (1.0 / m) * np.sum(dZL, axis=1, keepdims=True)

    dAl = np.dot(parameters['W' + str(L)].T, dZL)
    for l in reversed(range(1, L)):
        dZl = relu_backward(dAl, caches['Z' + str(l)])
        grads['dW' + str(l)] = (1.0 / m) * np.dot(dZl, caches['A' + str(l-1)].T)
        grads['db' + str(l)] = (1.0 / m) * np.sum(dZl, axis=1, keepdims=True)
        if l > 1:
            dAl = np.dot(parameters['W' + str(l)].T, dZl)
        
    return grads


In [8]:
## optimization and training loop

def update_parameters_sgd(parameters, grads, lr):
    L = len(parameters) // 2
    for l in range(1, L + 1):
        parameters['W' + str(l)] -= lr * grads['dW' + str(l)]
        parameters['b' + str(l)] -= lr * grads['db' + str(l)]
    return parameters

In [9]:
# create mini batches 
def create_mini_batches(X, y, batch_size = 64, seed = 42):
    np.random.seed(seed)
    m = X.shape[0]
    mini_batches = []

    permutation = list(np.random.permutation(m))
    shuffled_X = X[permutation, :]
    shuffled_y = y[permutation]

    num_complete_batches = m // batch_size
    for k in range(num_complete_batches):
        mini_batch_X = shuffled_X[k * batch_size: (k + 1) * batch_size, :]
        mini_batch_y = shuffled_y[k * batch_size: (k + 1) * batch_size]
        mini_batches.append((mini_batch_X, mini_batch_y))
    
    if m % batch_size != 0:
        mini_batch_X = shuffled_X[num_complete_batches * batch_size :, :]
        mini_batch_y = shuffled_y[num_complete_batches * batch_size :]
        mini_batches.append((mini_batch_X, mini_batch_y))

    return mini_batches



In [10]:


# def create_mini_batches(X, y, batch_size=64, seed=42):
#     np.random.seed(seed)
#     m = X.shape[0]
#     mini_batches = []
    
#     # Step 1: Shuffle X and y consistently
#     permutation = list(np.random.permutation(m))
#     shuffled_X = X[permutation, :]
#     shuffled_y = y[permutation]
    
#     # Step 2: Partition into mini-batches
#     num_complete_batches = m // batch_size
#     for k in range(num_complete_batches):
#         mini_batch_X = shuffled_X[k * batch_size : (k + 1) * batch_size, :]
#         mini_batch_y = shuffled_y[k * batch_size : (k + 1) * batch_size]
#         mini_batches.append((mini_batch_X, mini_batch_y))
        
#     # Handling the end case (residual batch if m is not divisible by batch_size)
#     if m % batch_size != 0:
#         mini_batch_X = shuffled_X[num_complete_batches * batch_size :, :]
#         mini_batch_y = shuffled_y[num_complete_batches * batch_size :]
#         mini_batches.append((mini_batch_X, mini_batch_y))
        
#     return mini_batches

In [11]:
## train model phase1

def train_model_phase1(X_train, y_train, layer_dims, epochs = 100, batch_size = 64, lr = 0.01):
    parameters = initialize_parameter_naive(layer_dims)

    for epoch in range(epochs):
        mini_batches = create_mini_batches(X_train, y_train, batch_size, seed = epoch)
        epoch_cost = 0

        for X_batch, y_batch in mini_batches:
            AL, caches = forward_propagation(X_batch, parameters)
            batch_cost = compute_cost(AL, y_batch)
            epoch_cost += batch_cost * (X_batch.shape[0] / X_train.shape[0])
            grads = backward_propagation(caches, parameters, y_batch)
            parameters = update_parameters_sgd(parameters, grads, lr)

        for epoch in range(epochs): 
            print(f"Epoch {epoch} -> Training Cost: {epoch_cost:.4f}")
        
    return parameters



In [ ]:
if __name__ == "__main__":
    print("Generating Synthetic dataset:")
    X_dummy = np.random.randn(2000, 3072)
    y_dummy = np.eye(3)[np.random.randint(0, 3, size=(2000,))]

    layer_dimensions = [3072, 128, 64, 3]
    
    trained_params = train_model_phase1(
        X_dummy, 
        y_dummy, 
        layer_dims=layer_dimensions, 
        epochs=50, 
        batch_size=64, 
        lr=0.05
    )
    print("Execution finished successfully!")


Phase 2 The Stabilizers